# Stage A — Deterministic Rule-Based Classification

This notebook preserves the original Version 1 rule baseline and implements Version 2 of the first stage of the hybrid GMRID risk-classification POC.

Stage A emits deterministic, explainable multi-label candidates. A prediction means that a high-confidence candidate was found; it does **not** mean that a multi-label record is completely resolved. Later stages may add or reject labels.


In [40]:
import ast
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    hamming_loss,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import MultiLabelBinarizer

RANDOM_SEED = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


In [41]:
train_poc = pd.read_csv(DATA_DIR / "train_poc.csv")
test_poc = pd.read_csv(DATA_DIR / "test_poc.csv")

def restore_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or value == "":
        return []
    return ast.literal_eval(value)

def to_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes"}

for frame in [train_poc, test_poc]:
    frame["true_risks"] = frame["true_risks"].apply(restore_list)
    frame["maritime_label"] = frame["maritime_label"].apply(to_bool)

# contains_port_info is intentionally unused: it is constant in the POC data.
print("Train shape:", train_poc.shape)
print("Test shape:", test_poc.shape)


Train shape: (2916, 16)
Test shape: (702, 16)


In [42]:
RISK_TAXONOMY = {
    "weather_disruption": "Weather Disruption",
    "natural_disaster": "Natural Disaster",
    "port_operational_disruption": "Port Operational Disruption",
    "port_closure": "Port Closure",
    "labor_strike_disruption": "Labor / Strike Disruption",
    "maritime_security_navigation_disruption": "Maritime Security / Navigation Disruption",
}
RISK_IDS = list(RISK_TAXONOMY)

mlb = MultiLabelBinarizer(classes=RISK_IDS)
mlb.fit([RISK_IDS])


,"classes classes: array-like of shape (n_classes,), default=NoneIndicates an ordering for the class labels.All entries should be unique (cannot contain duplicate classes).","['weather_disruption', 'natural_disaster', ...]"
,"sparse_output sparse_output: bool, default=FalseSet to True if output binary array is desired in CSR sparse format.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)A copy of the `classes` parameter when provided.Otherwise it corresponds to the sorted set of classes foundwhen fitting.","ndarray[object](6,)","['weather_disruption','natural_disaster','port_operational_disruption', 'port_closure','labor_strike_disruption', 'maritime_security_navigation_disruption']"


## Development and evaluation protocol

- **Development:** the original fixed 200-row training sample (`random_state=42`). It is used for rule design and error analysis.
- **Validation:** a fixed 600-row sample from the remaining training records (`random_state=43`). Version 1 and Version 2 are compared on these exact records.
- **Final test:** the original GMRID test split. It remains locked until the Version 2 rules and thresholds are frozen.

The historical Version 1 result remains documented: on the original 200-row development sample and original proxy mapping it achieved micro precision 0.653, micro recall 0.152, and micro F1 0.246. Since Version 2 corrects the category mapping and changes dataset membership, that historical number is not directly comparable to current results.


In [43]:
development_df = train_poc.sample(n=min(200, len(train_poc)), random_state=RANDOM_SEED).copy()
remaining_train = train_poc.drop(index=development_df.index)
validation_df = remaining_train.sample(
    n=min(600, len(remaining_train)),
    random_state=RANDOM_SEED + 1,
).copy()

development_df["dataset_split"] = "development"
validation_df["dataset_split"] = "validation"
test_poc = test_poc.copy()
test_poc["dataset_split"] = "final_test"

split_support = pd.DataFrame({
    "development": development_df["true_risks"].explode().value_counts(),
    "validation": validation_df["true_risks"].explode().value_counts(),
    "final_test": test_poc["true_risks"].explode().value_counts(),
}).reindex(RISK_IDS).fillna(0).astype(int)

assert set(development_df.index).isdisjoint(validation_df.index)
assert (split_support["validation"] > 0).all(), "Every risk must appear in validation."
display(split_support)


,development,validation,final_test
true_risks,,,
weather_disruption,24,91,109
natural_disaster,9,17,35
port_operational_disruption,92,296,351
port_closure,23,79,85
labor_strike_disruption,27,83,89
maritime_security_navigation_disruption,33,84,97


## Version 1 — preserved baseline

Version 1 uses literal substring matches, +3 per strong phrase, +1 per keyword, an optional +1 maritime metadata bonus, and one global threshold of 3. The unused `port` context and narrow exact phrases are preserved here so the validation comparison is faithful to the original method.


In [44]:
V1_RULES = {
    "weather_disruption": {
        "strong_phrases": ["severe weather", "weather advisory", "tropical cyclone", "storm surge", "high winds", "heavy rainfall", "severe winds", "heavy rain"],
        "keywords": ["storm", "flood", "flooding", "cyclone", "typhoon", "hurricane", "weather"],
        "context": None,
    },
    "natural_disaster": {
        "strong_phrases": ["major earthquake", "strong earthquake", "volcanic eruption", "tsunami warning"],
        "keywords": ["earthquake", "tsunami", "volcano", "volcanic", "landslide"],
        "context": None,
    },
    "port_operational_disruption": {
        "strong_phrases": ["port congestion", "terminal congestion", "port disruption", "cargo disruption", "container backlog", "vessel backlog", "berthing delays", "port delays"],
        "keywords": ["congestion", "backlog", "berthing delay", "cargo delay"],
        "context": "port",
    },
    "port_closure": {
        "strong_phrases": ["port closure", "port closed", "port is closed", "terminal closure", "terminal closed", "port operations suspended", "operations suspended", "harbor closed", "harbour closed"],
        "keywords": ["closure", "closed", "shutdown", "suspended"],
        "context": "port",
    },
    "labor_strike_disruption": {
        "strong_phrases": ["port strike", "dockworker strike", "dockworkers strike", "cargo strike", "industrial action", "labor dispute", "labour dispute", "workers strike", "worker strike"],
        "keywords": ["strike", "walkout", "dockworker", "dockworkers", "longshoremen", "union"],
        "context": None,
    },
    "maritime_security_navigation_disruption": {
        "strong_phrases": ["maritime advisory", "maritime security", "navigation warning", "navigation restriction", "waterway closure", "waterway disruption", "piracy attack", "piracy incident"],
        "keywords": ["piracy", "pirates", "navigation", "waterway"],
        "context": "maritime",
    },
}

V1_THRESHOLD = 3

def classify_v1(row):
    text = str(row.get("normalized_text", "")).lower()
    predictions = []
    for risk_id, config in V1_RULES.items():
        score = 3 * sum(phrase in text for phrase in config["strong_phrases"])
        score += sum(keyword in text for keyword in config["keywords"])
        if score > 0 and config.get("context") == "maritime" and row.get("maritime_label", False):
            score += 1
        if score >= V1_THRESHOLD:
            predictions.append(risk_id)
    return predictions


## Version 2 — contextual and explainable rules

Version 2 uses regex word boundaries, evidence families, overlap deduplication, explicit textual context, and risk-specific thresholds. Strong evidence contributes 3 points, supporting evidence contributes 1 point, textual context contributes 1 point where required, and `maritime_label` can contribute only one supporting point. Generic `Maritime Advisory` wording and generic delays are not sufficient maritime-security/navigation evidence.


In [45]:
def evidence(rule_id, label, pattern, weight):
    return {"rule_id": rule_id, "label": label, "pattern": pattern, "weight": weight}

PORT_CONTEXT = r"\b(?:port|ports|terminal|terminals|berth|berths|pier|piers|harbou?r|pilotage|container|cargo|vessel|vessels|shipside|waterside)\b"
LABOR_CONTEXT = r"\b(?:worker|workers|union|unions|dockworker|dockworkers|longshoremen|seamen|crew|tugboat|ferry|port|ports|cargo|transport|maritime|shipping)\b"
MARITIME_CONTEXT = r"\b(?:maritime|marine|ship|ships|shipping|vessel|vessels|tanker|channel|canal|strait|waterway|sea|ocean|navigation|port|harbou?r)\b"

V2_RULES = {
    "weather_disruption": {
        "threshold": 3,
        "context_required": False,
        "context_patterns": [],
        "strong": [
            evidence("severe_wind", "severe/strong/high wind", r"\b(?:severe|strong|high) winds?\b", 3),
            evidence("wind_warning", "wind or gale warning", r"\b(?:wind|gale) warnings?\b|\bgales?\b", 3),
            evidence("severe_weather", "severe/bad weather", r"\b(?:severe|bad|adverse|inclement) weather\b", 3),
            evidence("heavy_rain", "heavy rain/rainfall", r"\bheavy (?:rain|rains|rainfall)\b", 3),
            evidence("cyclone_family", "tropical cyclone/storm/depression", r"\btropical (?:cyclone|storm|depression)\b", 3),
            evidence("named_weather", "hurricane/typhoon/thunderstorm/tornado/hail", r"\b(?:hurricanes?|typhoons?|thunderstorms?|tornado(?:es)?|hail(?:stones?)?)\b", 3),
            evidence("flood", "flood/flooding", r"\bflood(?:ing|waters?)?\b", 3),
            evidence("storm_surge", "storm surge", r"\bstorm surge\b", 3),
        ],
        "supporting": [
            evidence("weather", "weather", r"\bweather\b", 1),
            evidence("storm", "storm", r"\bstorms?\b", 1),
            evidence("wind", "wind", r"\bwinds?\b", 1),
            evidence("rain", "rain", r"\brains?\b|\brainfall\b", 1),
        ],
    },
    "natural_disaster": {
        "threshold": 3,
        "context_required": False,
        "context_patterns": [],
        "strong": [
            evidence("earthquake", "earthquake/quake", r"\b(?:earthquakes?|quake)\b", 3),
            evidence("tsunami", "tsunami", r"\btsunami\b", 3),
            evidence("landslide", "landslide", r"\blandslides?\b", 3),
            evidence("volcanic_eruption", "volcanic eruption", r"\bvolcanic eruption\b", 3),
        ],
        "supporting": [
            evidence("magnitude", "reported magnitude", r"\bmagnitude \d+(?:\.\d+)?\b", 1),
            evidence("volcano", "volcano/volcanic", r"\bvolcan(?:o|oes|ic)\b", 1),
        ],
    },
    "port_operational_disruption": {
        "threshold": 4,
        "context_required": True,
        "context_patterns": [PORT_CONTEXT],
        "strong": [
            evidence("congestion", "port/berth/container congestion", r"\b(?:port|terminal|berth|container|vessel|fishing boats?) (?:congestion|congested)\b|\b(?:congestion|congested)\b", 3),
            evidence("backlog", "container/vessel backlog", r"\b(?:container|vessel|cargo) backlogs?\b", 3),
            evidence("waiting_time", "vessel/berth waiting time", r"\b(?:average )?waiting times?\b", 3),
            evidence("vessel_delay", "vessel/berthing/cargo delay", r"\b(?:vessel(?:'s)?|vessels|berthing|cargo|loading|unloading) delays?\b", 3),
            evidence("restricted_berthing", "restricted berthing", r"\brestricted berthing(?: capabilities)?\b", 3),
            evidence("operational_constraints", "operational constraints", r"\boperational constraints?\b", 3),
            evidence("reduced_capacity", "reduced/capacity limitation", r"\b(?:reduced capacity|capacity limitations?)\b", 3),
            evidence("operations_disrupted", "operations disrupted/resumed", r"\b(?:port |terminal |cargo |shipside |waterside )?operations? (?:are |were |have |has |had |may |might |likely to )?(?:disrupted|resume[ds]?|delayed|impacted|affected)\b", 3),
            evidence("contextual_operations", "contextual operational disruption", r"\b(?:port|terminal|cargo|shipside|waterside)\b(?:\W+\w+){0,7}\W+\boperations?\b(?:\W+\w+){0,5}\W+\b(?:disrupt(?:ed|ion)|resume[ds]?|delay(?:ed|s)?|impact(?:ed|s)?|affect(?:ed|s)?)\b", 3),
            evidence("port_disruption", "port/cargo/terminal disruption", r"\b(?:port|cargo|terminal) disruptions?\b", 3),
        ],
        "supporting": [
            evidence("delay", "delay", r"\bdelays?\b", 1),
            evidence("capacity", "capacity", r"\bcapacity\b", 1),
            evidence("operational", "operational", r"\boperational\b", 1),
        ],
    },
    "port_closure": {
        "threshold": 4,
        "context_required": True,
        "context_patterns": [PORT_CONTEXT, r"\b(?:ship )?channels?\b"],
        "strong": [
            evidence("context_then_close", "maritime facility closure", r"\b(?:port|terminal|berth|pier|harbou?r|pilotage|(?:ship )?channel)\b(?:\W+\w+){0,6}\W+\b(?:clos(?:e[ds]?|ure)|shut(?:s)? down|halt(?:ed|s)?|suspend(?:ed|s|ing)?)\b", 3),
            evidence("close_then_context", "closure of maritime facility", r"\b(?:clos(?:e[ds]?|ure)|shut(?:s)? down|halt(?:ed|s)?|suspend(?:ed|s|ing)?)\b(?:\W+\w+){0,6}\W+\b(?:port|terminal|berth|pier|harbou?r|pilotage|operations?|(?:ship )?channel)\b", 3),
            evidence("operations_closed", "operations halted/suspended", r"\boperations? (?:are |were |was |be |been |have been |has been )?(?:closed|halted|suspended|shut down)\b", 3),
            evidence("pilotage_suspended", "pilotage suspended", r"\bpilotage (?:is |was |has been )?suspended\b", 3),
        ],
        "supporting": [
            evidence("closure_term", "closure/closed/suspended/halted", r"\b(?:closure|closed|closes|suspended|suspend|halted|shutdown)\b", 1),
        ],
    },
    "labor_strike_disruption": {
        "threshold": 4,
        "context_required": True,
        "context_patterns": [LABOR_CONTEXT],
        "strong": [
            evidence("strike", "strike action", r"\b(?:strike|strikes|striking)\b", 3),
            evidence("threaten_strike", "threatened/threaten to strike", r"\bthreaten(?:ed|s|ing)? to strike\b", 3),
            evidence("industrial_action", "industrial action", r"\bindustrial action\b", 3),
            evidence("walkout", "walkout", r"\bwalkouts?\b", 3),
            evidence("union_action", "union action", r"\bunion action\b", 3),
            evidence("ferry_strike", "ferry strike", r"\bferry strike\b", 3),
        ],
        "supporting": [
            evidence("labor_group", "workers/union/seamen/tugboat workers", r"\b(?:workers?|unions?|seamen|dockworkers?|longshoremen|tugboat workers?)\b", 1),
            evidence("labor_dispute", "labor/labour dispute", r"\blabou?r disputes?\b", 1),
        ],
    },
    "maritime_security_navigation_disruption": {
        "threshold": 4,
        "context_required": True,
        "context_patterns": [MARITIME_CONTEXT],
        "strong": [
            evidence("piracy", "piracy/pirates", r"\b(?:piracy|pirates?)\b", 3),
            evidence("armed_robbery", "armed robbery/robbers aboard", r"\barmed (?:robbery|robbers?)\b(?:\W+\w+){0,5}\W+\b(?:aboard|ship|vessel|tanker)\b|\barmed robbers? aboard\b", 3),
            evidence("ship_channel_closure", "ship-channel closure", r"\bship channel\b(?:\W+\w+){0,5}\W+\b(?:clos(?:e[ds]?|ure)|shut|suspend(?:ed|s)?)\b", 3),
            evidence("shipping_channel_closure", "shipping-channel closure", r"\b(?:ship|shipping) channel\b(?:\W+\w+){0,12}\W+\b(?:clos(?:e[ds]?|ure)|shut|suspend(?:ed|s)?)\b", 3),
            evidence("waterway_disruption", "canal/strait/waterway disruption", r"\b(?:canal|strait|waterway)\b(?:\W+\w+){0,5}\W+\b(?:disrupt(?:ed|ion)|clos(?:e[ds]?|ure)|restrict(?:ed|ion))\b", 3),
            evidence("ship_attack", "ship/vessel/tanker attack", r"\b(?:cargo |container )?(?:ship|vessel|tanker)\b(?:\W+\w+){0,8}\W+\battack(?:ed|s)?\b|\battack(?:ed|s)?\b(?:\W+\w+){0,8}\W+\b(?:ship|vessel|tanker)\b", 3),
            evidence("ship_theft", "theft from ship/vessel", r"\b(?:ship|vessel) stores? theft\b|\btheft\b(?:\W+\w+){0,8}\W+\b(?:aboard|ship|vessel|anchorage)\b", 3),
            evidence("admission_restriction", "vessel-admission restriction", r"\b(?:vessel|ship) admission restrictions?\b|\brestrictions? on vessel admission\b", 3),
            evidence("navigation_restriction", "navigation restriction/warning", r"\bnavigation(?:al)? (?:restrictions?|warnings?)\b", 3),
            evidence("security_rerouting", "security-related rerouting", r"\b(?:security|attack|threat)\b(?:\W+\w+){0,6}\W+\b(?:rerout(?:e|ed|ing)|divert(?:ed|ing))\b", 3),
        ],
        "supporting": [
            evidence("maritime_advisory", "maritime advisory", r"\bmaritime advisory\b", 1),
            evidence("navigation", "navigation", r"\bnavigation\b", 1),
            evidence("security", "maritime security", r"\bmaritime security\b", 1),
        ],
    },
}


In [46]:
def normalize_for_rules(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"[-–—/]", " ", text)
    text = re.sub(r"[^\w\s']", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def headline_details_text(row):
    headline = normalize_for_rules(row.get("Headline", ""))
    details = normalize_for_rules(row.get("Details", ""))
    return " ".join(part for part in [headline, details] if part).strip()

def collect_evidence(text, rules, occupied_spans=None):
    occupied_spans = list(occupied_spans or [])
    matches = []
    for rule in rules:
        found = list(re.finditer(rule["pattern"], text, flags=re.IGNORECASE))
        usable = None
        for match in found:
            span = match.span()
            if not any(span[0] < old[1] and old[0] < span[1] for old in occupied_spans):
                usable = match
                break
        if usable is not None:
            occupied_spans.append(usable.span())
            matches.append({
                "rule_id": rule["rule_id"],
                "label": rule["label"],
                "match": usable.group(0),
                "weight": rule["weight"],
                "span": list(usable.span()),
            })
    return matches, occupied_spans

def score_v2_risk(row, risk_id):
    text = headline_details_text(row)
    config = V2_RULES[risk_id]

    strong, occupied = collect_evidence(text, config["strong"])
    supporting, _ = collect_evidence(text, config["supporting"], occupied)

    context_matches = []
    for pattern in config["context_patterns"]:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            context_matches.append(match.group(0))

    has_evidence = bool(strong or supporting)
    context_satisfied = (not config["context_required"]) or bool(context_matches)
    context_bonus = 1 if has_evidence and config["context_required"] and context_satisfied else 0
    maritime_metadata_bonus = (
        1
        if risk_id == "maritime_security_navigation_disruption"
        and has_evidence
        and bool(row.get("maritime_label", False))
        else 0
    )

    evidence_score = sum(item["weight"] for item in strong + supporting)
    score = evidence_score + context_bonus + maritime_metadata_bonus
    predicted = score >= config["threshold"] and context_satisfied

    if predicted:
        reason = f"predicted: score {score} >= {config['threshold']} with required context satisfied"
    elif not has_evidence:
        reason = "rejected: no rule evidence"
    elif not context_satisfied:
        reason = "rejected: evidence found but required context missing"
    else:
        reason = f"rejected: score {score} < {config['threshold']}"

    return {
        "score": score,
        "threshold": config["threshold"],
        "matched_strong_evidence": strong,
        "matched_supporting_evidence": supporting,
        "context_matches": sorted(set(context_matches)),
        "context_bonus": context_bonus,
        "maritime_metadata_bonus": maritime_metadata_bonus,
        "context_required": config["context_required"],
        "context_satisfied": context_satisfied,
        "predicted": predicted,
        "reason": reason,
    }

def classify_v2(row):
    explanations = {risk_id: score_v2_risk(row, risk_id) for risk_id in RISK_IDS}
    predictions = [risk_id for risk_id in RISK_IDS if explanations[risk_id]["predicted"]]
    if len(predictions) == 0:
        status = "no_stage_a_prediction"
    elif len(predictions) == 1:
        status = "stage_a_candidate_found"
    else:
        status = "multiple_stage_a_candidates"
    return {
        "stage_a_predictions": predictions,
        "stage_a_scores": {risk_id: explanations[risk_id]["score"] for risk_id in RISK_IDS},
        "stage_a_explanations": explanations,
        "stage_a_status": status,
    }

def apply_v2(frame):
    output = frame.copy()
    result = output.apply(classify_v2, axis=1)
    output["stage_a_predictions"] = result.apply(lambda item: item["stage_a_predictions"])
    output["stage_a_scores"] = result.apply(lambda item: item["stage_a_scores"])
    output["stage_a_explanations"] = result.apply(lambda item: item["stage_a_explanations"])
    output["stage_a_status"] = result.apply(lambda item: item["stage_a_status"])
    return output


In [47]:
def evaluate_predictions(frame, prediction_column, dataset_split, version):
    y_true = mlb.transform(frame["true_risks"])
    y_pred = mlb.transform(frame[prediction_column])

    per_precision, per_recall, per_f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    predicted_support = y_pred.sum(axis=0)
    true_positives = (y_true & y_pred).sum(axis=0)
    false_positives = ((1 - y_true) & y_pred).sum(axis=0)
    false_negatives = (y_true & (1 - y_pred)).sum(axis=0)

    per_risk = pd.DataFrame({
        "dataset_split": dataset_split,
        "version": version,
        "risk": RISK_IDS,
        "precision": per_precision,
        "recall": per_recall,
        "f1": per_f1,
        "support": support.astype(int),
        "predicted_count": predicted_support.astype(int),
        "true_positives": true_positives.astype(int),
        "false_positives": false_positives.astype(int),
        "false_negatives": false_negatives.astype(int),
    })

    summary_rows = []
    for average in ["micro", "macro", "weighted"]:
        p, r, f, _ = precision_recall_fscore_support(
            y_true, y_pred, average=average, zero_division=0
        )
        summary_rows.append({
            "dataset_split": dataset_split,
            "version": version,
            "average": average,
            "precision": p,
            "recall": r,
            "f1": f,
        })

    no_prediction = frame[prediction_column].apply(len).eq(0)
    coverage = 1 - no_prediction.mean()
    extras = {
        "dataset_split": dataset_split,
        "version": version,
        "records": len(frame),
        "exact_match_ratio": accuracy_score(y_true, y_pred),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "prediction_coverage": coverage,
        "no_prediction_count": int(no_prediction.sum()),
        "no_prediction_percentage": 100 * no_prediction.mean(),
    }
    return per_risk, pd.DataFrame(summary_rows), pd.DataFrame([extras])


## Development and validation evaluation

These cells compare Version 1 and the frozen Version 2 configuration on the same records. Version 2 thresholds are fixed at 3 for weather/natural-disaster rules and 4 for context-dependent risks.


In [48]:
development_df["v1_predictions"] = development_df.apply(classify_v1, axis=1)
validation_df["v1_predictions"] = validation_df.apply(classify_v1, axis=1)
development_v2 = apply_v2(development_df)
validation_v2 = apply_v2(validation_df)

evaluation_parts = []
summary_parts = []
extra_parts = []
for frame, column, split, version in [
    (development_df, "v1_predictions", "development", "Version 1"),
    (development_v2, "stage_a_predictions", "development", "Version 2"),
    (validation_df, "v1_predictions", "validation", "Version 1"),
    (validation_v2, "stage_a_predictions", "validation", "Version 2"),
]:
    per_risk, summary, extras = evaluate_predictions(frame, column, split, version)
    evaluation_parts.append(per_risk)
    summary_parts.append(summary)
    extra_parts.append(extras)

per_risk_results = pd.concat(evaluation_parts, ignore_index=True)
aggregate_results = pd.concat(summary_parts, ignore_index=True)
coverage_results = pd.concat(extra_parts, ignore_index=True)

display(aggregate_results.round(3))
display(coverage_results.round(3))
display(per_risk_results.round(3))


,dataset_split,version,average,precision,recall,f1
0,development,Version 1,micro,0.651,0.135,0.223
1,development,Version 1,macro,0.633,0.181,0.253
2,development,Version 1,weighted,0.702,0.135,0.197
3,development,Version 2,micro,0.668,0.620,0.643
4,development,Version 2,macro,0.730,0.690,0.626
5,development,Version 2,weighted,0.796,0.620,0.630
6,validation,Version 1,micro,0.686,0.168,0.269
7,validation,Version 1,macro,0.821,0.194,0.263
8,validation,Version 1,weighted,0.789,0.168,0.233
9,validation,Version 2,micro,0.669,0.612,0.639


,dataset_split,version,records,exact_match_ratio,hamming_loss,prediction_coverage,no_prediction_count,no_prediction_percentage
0,development,Version 1,200,0.140,0.162,0.215,157,78.500
1,development,Version 2,200,0.445,0.119,0.730,54,27.000
2,validation,Version 1,600,0.158,0.164,0.258,445,74.167
3,validation,Version 2,600,0.453,0.125,0.757,146,24.333


,dataset_split,version,risk,precision,recall,f1,support,predicted_count,true_positives,false_positives,false_negatives
0,development,Version 1,weather_disruption,0.588,0.417,0.488,24,17,10,7,14
1,development,Version 1,natural_disaster,1.000,0.222,0.364,9,2,2,0,7
2,development,Version 1,port_operational_disruption,1.000,0.076,0.141,92,7,7,0,85
3,development,Version 1,port_closure,0.462,0.261,0.333,23,13,6,7,17
4,development,Version 1,labor_strike_disruption,0.750,0.111,0.194,27,4,3,1,24
5,development,Version 1,maritime_security_navigation_disruption,0.000,0.000,0.000,33,0,0,0,33
6,development,Version 2,weather_disruption,0.362,0.875,0.512,24,58,21,37,3
7,development,Version 2,natural_disaster,0.818,1.000,0.900,9,11,9,2,0
8,development,Version 2,port_operational_disruption,0.935,0.630,0.753,92,62,58,4,34
9,development,Version 2,port_closure,0.512,0.913,0.656,23,41,21,20,2


## Locked final test evaluation

Run only after the Version 2 rules and thresholds above are frozen. Do not inspect individual test errors or revise rules after viewing these aggregate results.


In [49]:
test_v2 = apply_v2(test_poc)
test_per_risk, test_aggregate, test_coverage = evaluate_predictions(
    test_v2, "stage_a_predictions", "final_test", "Version 2"
)

per_risk_results = pd.concat([per_risk_results, test_per_risk], ignore_index=True)
aggregate_results = pd.concat([aggregate_results, test_aggregate], ignore_index=True)
coverage_results = pd.concat([coverage_results, test_coverage], ignore_index=True)

display(test_aggregate.round(3))
display(test_coverage.round(3))
display(test_per_risk.round(3))


,dataset_split,version,average,precision,recall,f1
0,final_test,Version 2,micro,0.646,0.625,0.635
1,final_test,Version 2,macro,0.730,0.679,0.625
2,final_test,Version 2,weighted,0.766,0.625,0.625


,dataset_split,version,records,exact_match_ratio,hamming_loss,prediction_coverage,no_prediction_count,no_prediction_percentage
0,final_test,Version 2,702,0.457,0.131,0.783,152,21.652


,dataset_split,version,risk,precision,recall,f1,support,predicted_count,true_positives,false_positives,false_negatives
0,final_test,Version 2,weather_disruption,0.417,0.917,0.573,109,240,100,140,9
1,final_test,Version 2,natural_disaster,0.850,0.971,0.907,35,40,34,6,1
2,final_test,Version 2,port_operational_disruption,0.872,0.581,0.697,351,234,204,30,147
3,final_test,Version 2,port_closure,0.442,0.718,0.547,85,138,61,77,24
4,final_test,Version 2,labor_strike_disruption,0.885,0.775,0.826,89,78,69,9,20
5,final_test,Version 2,maritime_security_navigation_disruption,0.917,0.113,0.202,97,12,11,1,86


## Prediction and error exports

Explanations are stored for every training and test record. Error analysis includes complete validation and final-test label errors; no test error is manually inspected or used for tuning.


In [50]:
# Score all training records for a complete reproducible prediction export.
all_train_v2 = train_poc.copy()
all_train_v2["dataset_split"] = "unused_train"
all_train_v2.loc[development_df.index, "dataset_split"] = "development"
all_train_v2.loc[validation_df.index, "dataset_split"] = "validation"
all_train_v2 = apply_v2(all_train_v2)

prediction_columns = [
    "dataset_split", "id", "Headline", "Category", "true_risks",
    "stage_a_predictions", "stage_a_status", "stage_a_scores", "stage_a_explanations",
]
prediction_export = pd.concat(
    [all_train_v2[prediction_columns], test_v2[prediction_columns]],
    ignore_index=True,
).copy()

for column in ["true_risks", "stage_a_predictions", "stage_a_scores", "stage_a_explanations"]:
    prediction_export[column] = prediction_export[column].apply(
        lambda value: json.dumps(value, ensure_ascii=False, sort_keys=True)
    )

prediction_path = OUTPUT_DIR / "stage_a_v2_predictions.csv"
prediction_export.to_csv(prediction_path, index=False)
print("Saved:", prediction_path, "rows:", len(prediction_export))


Saved: c:\Users\prakh\OneDrive\Documents\INTERNSHIP WORK\gmrid-risk-classification-project\outputs\stage_a_v2_predictions.csv rows: 3618


In [51]:
def build_error_analysis(frame, dataset_split):
    rows = []
    for row_index, row in frame.iterrows():
        true_set = set(row["true_risks"])
        predicted_set = set(row["stage_a_predictions"])
        for risk_id in RISK_IDS:
            if risk_id in predicted_set and risk_id not in true_set:
                error_type = "false_positive"
            elif risk_id in true_set and risk_id not in predicted_set:
                error_type = "false_negative"
            else:
                continue

            risk_explanation = row["stage_a_explanations"][risk_id]
            rows.append({
                "dataset_split": dataset_split,
                "row_index": row_index,
                "id": row["id"],
                "inspected_risk": risk_id,
                "error_type": error_type,
                "selection": "complete",
                "Headline": row["Headline"],
                "Details": row["Details"],
                "Category": row["Category"],
                "Summarized_label": row["Summarized_label"],
                "maritime_label": row["maritime_label"],
                "true_risks": json.dumps(row["true_risks"], ensure_ascii=False),
                "stage_a_predictions": json.dumps(row["stage_a_predictions"], ensure_ascii=False),
                "risk_score": risk_explanation["score"],
                "matched_strong_evidence": json.dumps(risk_explanation["matched_strong_evidence"], ensure_ascii=False),
                "matched_supporting_evidence": json.dumps(risk_explanation["matched_supporting_evidence"], ensure_ascii=False),
                "applied_context_bonuses": json.dumps({
                    "text_context": risk_explanation["context_bonus"],
                    "maritime_metadata": risk_explanation["maritime_metadata_bonus"],
                    "context_matches": risk_explanation["context_matches"],
                }, ensure_ascii=False),
                "prediction_explanation": risk_explanation["reason"],
            })
    return pd.DataFrame(rows)

# Development and validation are suitable for manual rule analysis.
# Final-test errors are exported for record-keeping only and were not used to tune V2.
error_analysis = pd.concat([
    build_error_analysis(development_v2, "development"),
    build_error_analysis(validation_v2, "validation"),
    build_error_analysis(test_v2, "final_test"),
], ignore_index=True)

error_path = OUTPUT_DIR / "stage_a_error_analysis.csv"
error_analysis.to_csv(error_path, index=False)

per_risk_path = OUTPUT_DIR / "stage_a_metrics_per_risk.csv"
aggregate_path = OUTPUT_DIR / "stage_a_metrics_aggregate.csv"
coverage_path = OUTPUT_DIR / "stage_a_metrics_coverage.csv"
per_risk_results.to_csv(per_risk_path, index=False)
aggregate_results.to_csv(aggregate_path, index=False)
coverage_results.to_csv(coverage_path, index=False)

print("Saved:", error_path, "rows:", len(error_analysis))
print("Saved metric tables to:", OUTPUT_DIR)


Saved: c:\Users\prakh\OneDrive\Documents\INTERNSHIP WORK\gmrid-risk-classification-project\outputs\stage_a_error_analysis.csv rows: 1142
Saved metric tables to: c:\Users\prakh\OneDrive\Documents\INTERNSHIP WORK\gmrid-risk-classification-project\outputs


In [52]:
# Reproducibility and leakage guards.
train_ids = set(train_poc["id"])
test_ids = set(test_poc["id"])
train_text = set(train_poc["normalized_text"].fillna(""))
test_text = set(test_poc["normalized_text"].fillna(""))

assert train_ids.isdisjoint(test_ids)
assert train_text.isdisjoint(test_text)
assert len(prediction_export) == len(train_poc) + len(test_poc)
assert set(error_analysis["selection"]) == {"complete"}

print("Train/test ID overlap: 0")
print("Train/test exact normalized-text overlap: 0")
print("Stage A Version 2 execution complete.")


Train/test ID overlap: 0
Train/test exact normalized-text overlap: 0
Stage A Version 2 execution complete.
